### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
- Traking agent behabior with logging, analytics and debugging
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")

#### Summarization Middleware
Automatically summarize conversation history when approching token limits, preversing recent messages while compressing older contenxt. Summarization is useful for the following:
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [12]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent = create_agent(
    model = "gpt-4o-mini",
    checkpointer = InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = "gpt-4o-mini",
            trigger= ("messages", 10),
            keep=("messages", 4)
        )
    ]
)

In [13]:
### Run with thread id
config = {"configurable": {"thread_id": "test-1"}}

In [15]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?"
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")


Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='1f32870b-fab3-4e75-97f6-15c3ee6a7fcd'), AIMessage(content='2 + 2 equals 4.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 14, 'total_tokens': 22, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_454234172d', 'id': 'chatcmpl-D7FJXBj2W1littpgH0H2vAhiGEEQV', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c4125-ff42-7cf3-8c8a-375744bbee5a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 8, 'total_tokens': 22, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': 

### Token Size

In [19]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city:str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, bussiness center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model = "gpt-4o-mini",
    tools = [search_hotels],
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 550),
            keep=("tokens", 200)
        ),
    ]
)
config = {"configurable" : {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token

In [20]:
# Run Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Mexico City"]

for city in cities:
    response = agent.invoke(
        {"messages":  [HumanMessage(content=f"Find hotels in {city}")]},
        config = config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response["messages"])} messages")
    print(f"{(response['messages'])}")

Paris: ~152 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='2c206d5c-4588-471b-a169-29d85fca62bd'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 53, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f4ae844694', 'id': 'chatcmpl-D7ZXxHZueiK9CJWFyNa7DRWoLVLfN', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c45c8-bb5b-7f51-b462-f1104d3fe8c2-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_bjG2kMBJupKfBgoxo6bMyD4K', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 53

### Fraction

In [25]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city:str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Gran Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing
agent = create_agent(
    model = "gpt-4o-mini",
    tools = [search_hotels],
    checkpointer = InMemorySaver(),
    middleware =[
        SummarizationMiddleware(
            model = "gpt-4o-mini",
            trigger = ("fraction", 0.005), # 0.5% = ~640 tokens
            keep = ("fraction", 0.002),    # 0.2% = ~256 tokens
        )
    ]
)

config = {"configurable" : {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Run Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Mexico City"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config = config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000 # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response["messages"])} msgs")
    print(response["messages"])

Paris: ~70 tokens (0.0547%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='32eea37c-1a0d-45f2-ab1b-5770f36749ce'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 44, 'total_tokens': 59, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_6c0d1490cb', 'id': 'chatcmpl-D7ZqMSbbtpQsBrkqUWISMOmedNhK9', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c45da-22ee-7442-b6be-1148299fc094-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_ongcPk5F8LugvRX8ajjMfRwG', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 44